# [7.1] Logit Lens, Tuned Lens, and Patchscopes - Exercises

Activation-to-language tools translate hidden activations into token predictions or prompt-conditioned answers. In this notebook you build the core mechanics and, just as importantly, the controls that stop decoded text from becoming a story you cannot defend.

```yaml
gt_tier: GT-1 activation-to-language preflight
exercise_id: 7_1_logit_lens_tuned_lens_and_patchscopes
expected_runtime: 60-90 minutes for CPU exercises; several minutes for CUDA GELU-1L preflight
requires_gpu: true for the TransformerLens preflight; false for the implementation exercises
```

<details>
<summary>Expected output</summary>

Each local test should print an "All tests ... passed" line. The final report-backed cells should show a pinned `gelu-1l` run where a ridge tuned lens improves held-out decoding and Patchscope activation insertion beats a text-only target prompt.

</details>

<details>
<summary>Help - how to read this section</summary>

A decoded token is a hypothesis about what a decoder can extract from an activation. Trust grows when held-out accuracy, text-only baselines, counterfactual activations, and random-activation controls all point in the same direction.

</details>


In [ ]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import torch as t
import torch.nn.functional as F

chapter = "chapter7_activation_to_language"
section = "part1_lenses_patchscopes"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_lenses_patchscopes.tests as tests

PatchscopeTemplate = Literal["entity", "next_token", "fact"]


@dataclass(frozen=True)
class LensAccuracyReport:
    logit_lens_accuracy: float
    tuned_lens_accuracy: float
    improvement: float
    tuned_lens_improves: bool


@dataclass(frozen=True)
class PatchscopeAccuracyReport:
    patchscope_accuracy: float
    text_only_accuracy: float
    improvement: float
    beats_text_only: bool


@dataclass(frozen=True)
class CounterfactualActivationReport:
    original_answer: int
    patched_answer: int
    changed: bool


@dataclass(frozen=True)
class RandomActivationConfidenceReport:
    mean_confidence: float
    max_confidence: float
    passes_low_confidence: bool


## 1. Logit Lens

Project residual activations directly through the unembedding. This is the cheapest readout and the baseline for later lenses.

<details>
<summary>Expected output</summary>

The controlled residual directions should decode to logits `[[2.0, 0.0, 1.0], [0.0, 3.0, 1.0]]` and top token ids `[[0], [1]]`.

```text
All tests in `test_logit_lens_and_top_tokens_match_reference` passed!
All tests in `test_top_tokens_rejects_invalid_k` passed!
```

</details>

<details>
<summary>Help - logit lens is a baseline</summary>

If a token is top-ranked here, it is linearly decodable through the unembedding. That does not prove the final model will keep the answer, or that this is the only representation of the information.

</details>

Common bug: applying softmax before selecting top logits. Use logits for ranking; probabilities are for reporting confidence.


In [ ]:
def logit_lens(residual_stream: t.Tensor, unembedding: t.Tensor) -> t.Tensor:
    raise NotImplementedError()


def top_tokens(logits: t.Tensor, *, k: int = 5) -> tuple[t.Tensor, t.Tensor]:
    raise NotImplementedError()


tests.test_logit_lens_and_top_tokens_match_reference(logit_lens, top_tokens)
tests.test_top_tokens_rejects_invalid_k(top_tokens)


## 2. Tuned Lens

A tuned lens applies a learned affine correction before decoding. It is useful only if it improves held-out decoding rather than memorizing the training set.

<details>
<summary>Expected output</summary>

Ordinary logit lens accuracy should be `0.0`, tuned-lens accuracy should be `1.0`, and the report should mark the tuned lens as an improvement.

```text
All tests in `test_tuned_lens_improves_over_logit_lens_on_toy_targets` passed!
All tests in `test_tuned_lens_uses_bias_and_leading_dims` passed!
All tests in `test_prediction_accuracy_rejects_shape_mismatch` passed!
```

</details>

<details>
<summary>Help - calibration needs held-out examples</summary>

A tuned lens is allowed to learn a decoder. That makes the train/held-out split part of the claim: without it, the lens may only have learned the examples you showed it.

</details>

Common bug: applying the bias after the unembedding, or comparing logits directly instead of measuring target-token accuracy.


In [ ]:
def tuned_lens(
    residual_stream: t.Tensor,
    lens_weight: t.Tensor,
    lens_bias: t.Tensor | None,
    unembedding: t.Tensor,
) -> t.Tensor:
    raise NotImplementedError()


def prediction_accuracy(logits: t.Tensor, target_token_ids: t.Tensor) -> float:
    raise NotImplementedError()


def lens_accuracy_report(
    logit_lens_logits: t.Tensor,
    tuned_lens_logits: t.Tensor,
    target_token_ids: t.Tensor,
) -> LensAccuracyReport:
    raise NotImplementedError()


tests.test_tuned_lens_improves_over_logit_lens_on_toy_targets(
    logit_lens,
    tuned_lens,
    lens_accuracy_report,
)
tests.test_tuned_lens_uses_bias_and_leading_dims(tuned_lens, prediction_accuracy)
tests.test_prediction_accuracy_rejects_shape_mismatch(prediction_accuracy)


## 3. Attention Lens

Attention lens decodes the value stream after applying the attention pattern. This asks what a query read, not merely what each value vector contained.

<details>
<summary>Expected output</summary>

The controlled attention pattern and values should decode to `[[[1.0, 0.0], [0.25, 1.5]]]`.

```text
All tests in `test_attention_lens_decodes_attention_weighted_values` passed!
All tests in `test_attention_lens_rejects_rank_or_key_mismatch` passed!
```

</details>

<details>
<summary>Help - watch the query/key axes</summary>

The multiplication is `attention_pattern @ value_vectors`. Mixing query and key axes can produce a tensor with plausible shape but the wrong interpretation.

</details>

Common bug: decoding raw value vectors without applying attention first.


In [ ]:
def attention_lens(
    attention_pattern: t.Tensor,
    value_vectors: t.Tensor,
    unembedding: t.Tensor,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_attention_lens_decodes_attention_weighted_values(attention_lens)
tests.test_attention_lens_rejects_rank_or_key_mismatch(attention_lens)


## 4. Patchscope Templates And Reports

Patchscopes insert source activations into a target prompt. The prompt wording is part of the artifact, so you must compare against the same prompt without the activation.

<details>
<summary>Expected output</summary>

The three templates should render with `<ACT>`, Patchscope accuracy should be `1.0`, text-only accuracy should be `0.5`, and the report should mark Patchscope as beating text only.

</details>

<details>
<summary>Help - prompt priors are a baseline</summary>

If the target prompt alone can answer correctly, the activation did not earn the credit. Keep the target ids identical for the patched and text-only paths.

</details>

Common bug: changing the template string or removing the placeholder, which breaks comparisons across runs.


In [ ]:
def patchscope_prompt(template: PatchscopeTemplate, placeholder: str = "<ACT>") -> str:
    raise NotImplementedError()


def patchscope_accuracy_report(
    patchscope_logits: t.Tensor,
    text_only_logits: t.Tensor,
    target_answer_ids: t.Tensor,
) -> PatchscopeAccuracyReport:
    raise NotImplementedError()


## 5. Activation Patching Helper

The helper replaces the final target-prompt activation with a source activation. Earlier positions should remain untouched.

<details>
<summary>Expected output</summary>

The final target-prompt activation should be replaced while earlier positions stay zero.

```text
All tests in `test_patchscope_templates_and_accuracy_report` passed!
All tests in `test_replace_final_position_activation_rejects_bad_shapes` passed!
```

</details>

<details>
<summary>Help - replace, do not append</summary>

Patchscopes compare the same target prompt with and without an inserted activation. Appending a new token changes the target prompt and invalidates the baseline.

</details>

Common bug: replacing every sequence position instead of only the final target position.


In [ ]:
def replace_final_position_activation(
    activations: t.Tensor,
    source_activation: t.Tensor,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_patchscope_templates_and_accuracy_report(
    patchscope_prompt,
    patchscope_accuracy_report,
    replace_final_position_activation,
)
tests.test_replace_final_position_activation_rejects_bad_shapes(
    replace_final_position_activation,
)


## 6. Counterfactual And Random Controls

A useful activation-conditioned decoder should react to counterfactual activations and should not become confident on random activations.

<details>
<summary>Expected output</summary>

The counterfactual toy logits should change answer `0 -> 1`, and uniform random logits over four tokens should have max confidence `0.25`.

```text
All tests in `test_counterfactual_and_random_activation_controls` passed!
```

</details>

<details>
<summary>Help - controls tell you what would falsify the story</summary>

No counterfactual change means the decoder may be insensitive to the activation. High random confidence means the prompt or decoder may have a strong answer prior.

</details>

Common bug: reporting the target-token probability instead of the maximum softmax confidence over all tokens.


In [ ]:
def counterfactual_activation_report(
    original_logits: t.Tensor,
    patched_logits: t.Tensor,
) -> CounterfactualActivationReport:
    raise NotImplementedError()


def random_activation_confidence_report(
    random_logits: t.Tensor,
    *,
    max_allowed_confidence: float = 0.6,
) -> RandomActivationConfidenceReport:
    raise NotImplementedError()


tests.test_counterfactual_and_random_activation_controls(
    counterfactual_activation_report,
    random_activation_confidence_report,
)


## Whole-Notebook Contract

Once all exercises pass, your implementation should satisfy the same local smoke-test contract as `solutions.py`.

<details>
<summary>Expected output</summary>

After uncommenting the last line, the test should print:

```text
All tests in `test_notebook_contract` passed!
```

</details>


In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    residual = t.tensor([[1.0, 0.0], [0.0, 1.0]])
    unembedding = t.tensor([[2.0, 0.0, 1.0], [0.0, 3.0, 1.0]])
    logits = logit_lens(residual, unembedding)
    top_ids, top_probs = top_tokens(logits, k=1)

    toy_residual = t.tensor([[1.0, 0.0], [0.0, 1.0]])
    toy_unembedding = t.eye(2)
    logit_logits = logit_lens(toy_residual, toy_unembedding)
    tuned_logits = tuned_lens(
        toy_residual,
        t.tensor([[0.0, 1.0], [1.0, 0.0]]),
        None,
        toy_unembedding,
    )
    targets = t.tensor([1, 0])

    patchscope_logits = t.tensor([[2.0, 0.0], [0.0, 2.0]])
    text_only_logits = t.tensor([[0.0, 2.0], [0.0, 2.0]])
    patch_targets = t.tensor([0, 1])
    patched_acts = replace_final_position_activation(
        t.zeros(1, 3, 2),
        t.tensor([1.0, -1.0]),
    )

    return {
        "logit_lens": {
            "logits": logits.tolist(),
            "top_ids": top_ids.tolist(),
            "top_probs": top_probs.tolist(),
        },
        "tuned_lens": lens_accuracy_report(
            logit_logits,
            tuned_logits,
            targets,
        ).__dict__,
        "attention_lens": {
            "logits": attention_lens(
                t.tensor([[[1.0, 0.0], [0.0, 1.0]]]),
                t.tensor([[[1.0, 0.0], [0.0, 1.0]]]),
                t.eye(2),
            ).tolist(),
        },
        "patchscope": {
            "entity_prompt": patchscope_prompt("entity"),
            "next_token_prompt": patchscope_prompt("next_token"),
            "fact_prompt": patchscope_prompt("fact"),
            "patched_final_activation": patched_acts[0, -1].tolist(),
            **patchscope_accuracy_report(
                patchscope_logits,
                text_only_logits,
                patch_targets,
            ).__dict__,
        },
        "counterfactual": counterfactual_activation_report(
            t.tensor([2.0, 0.0]),
            t.tensor([0.0, 3.0]),
        ).__dict__,
        "random_confidence": random_activation_confidence_report(
            t.zeros(3, 4),
            max_allowed_confidence=0.3,
        ).__dict__,
    }


# Uncomment after finishing all exercises.
# tests.test_notebook_contract(run_smoke_test)


## Signature Result

The final result is report-backed and uses the committed CUDA evidence. It does not rerun TransformerLens inside the notebook; rerun `solutions.run_gpu_test(max_vram_gb=24.0)` from Python when you want to refresh the report.

<details>
<summary>Expected output</summary>

The table should show `gelu-1l`, 16 train prompts, 6 held-out prompts, 40 held-out positions, logit-lens accuracy around `0.075`, tuned-lens accuracy around `0.450`, Patchscope/text-only accuracy `1.000 / 0.000`, random max confidence around `0.0407`, and peak VRAM under `1 GB`.

</details>

<details>
<summary>Interpreting the signature result</summary>

The tuned lens improves agreement with the model's own final predictions on held-out positions. Patchscope activation insertion recovers the source prompt's final-token prediction on all six controlled pairs while the neutral text-only prompt recovers none. This is a mechanics preflight, not a semantic truth benchmark.

</details>


In [ ]:
def _load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


def signature_table(gpu: dict) -> list[tuple[str, object]]:
    return [
        ("model", f"{gpu['model_name']} / {gpu['hf_model_id']}"),
        ("HF revision", gpu["hf_revision"][:12]),
        ("train / held-out prompts", f"{gpu['train_prompt_count']} / {gpu['heldout_prompt_count']}"),
        ("held-out positions", gpu["heldout_position_count"]),
        ("logit-lens accuracy", round(gpu["logit_lens_accuracy"], 3)),
        ("tuned-lens accuracy", round(gpu["tuned_lens_accuracy"], 3)),
        ("tuned-lens improvement", round(gpu["tuned_lens_improvement"], 3)),
        ("final decode max abs error", f"{gpu['final_decode_max_abs_error']:.2e}"),
        ("Patchscope hook", gpu["patchscope_hook_name"]),
        ("Patchscope / text-only accuracy", f"{gpu['patchscope_accuracy']:.3f} / {gpu['text_only_accuracy']:.3f}"),
        ("patched min target margin", round(gpu["patchscope_min_patched_target_margin"], 4)),
        ("text-only max target margin", round(gpu["patchscope_max_text_only_target_margin"], 3)),
        ("counterfactual decoded token", f"{gpu['counterfactual_original_token']!r} -> {gpu['counterfactual_patched_token']!r}"),
        ("random max confidence", round(gpu["random_max_confidence"], 4)),
        ("attention lens shape", gpu["attention_lens_logits_shape"]),
        ("peak VRAM GB", round(gpu["peak_vram_gb"], 3)),
    ]


gpu = run_gpu_test(max_vram_gb=24.0)
signature_table(gpu)


In [ ]:
import matplotlib.pyplot as plt

gpu = run_gpu_test(max_vram_gb=24.0)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))

axes[0].bar(
    ["logit", "tuned"],
    [gpu["logit_lens_accuracy"], gpu["tuned_lens_accuracy"]],
    color=["#2563eb", "#16a34a"],
)
axes[0].set_title("Held-out lens decoding")
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("top-1 agreement")

axes[1].bar(
    ["text-only", "patched"],
    [gpu["text_only_accuracy"], gpu["patchscope_accuracy"]],
    color=["#94a3b8", "#7c3aed"],
)
axes[1].set_title("Patchscope target recovery")
axes[1].set_ylim(0, 1)

axes[2].bar(
    ["decode err", "random conf", "patch margin"],
    [
        gpu["final_decode_max_abs_error"] / 1e-4,
        gpu["random_max_confidence"] / 0.1,
        gpu["patchscope_min_patched_target_margin"] / 0.01,
    ],
    color=["#0f766e", "#f97316", "#16a34a"],
)
axes[2].axhline(1.0, color="#334155", linewidth=1, linestyle="--")
axes[2].set_title("Control thresholds")
axes[2].set_ylabel("actual / threshold")
axes[2].tick_params(axis="x", rotation=15)

fig.tight_layout()
plt.show()


## Limitations

The local tests use tiny tensors. The CUDA report uses one small public TransformerLens checkpoint, safe generated prompts, and aggregate metrics. Targets are the model's own final-logit argmax predictions, not human semantic labels. This is not a released tuned-lens benchmark, not proof that decoded text is the model's belief, and not broad Patchscope validation across model families, layers, templates, or tasks.
